# 01 - Bitext dataset exploration

Run from the project environment. This notebook audits raw data without training a model or changing labels. The standard-library pipeline also runs without Jupyter.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the project or notebooks directory")
sys.path.insert(0, str(ROOT))
from src.data.inspect import inspect_csv, write_report

## Inspect the raw dataset

First run `python -m src.data.download` from the project root. The pinned source and file hashes are saved in `data/raw/manifest.json`.

In [ ]:
summary = inspect_csv(ROOT / "data/raw/bitext.csv")
{k: summary[k] for k in ["rows", "columns", "missing_by_column", "word_count"]}

In [ ]:
summary["category_counts"], summary["intent_counts"]

In [ ]:
{k: summary[k] for k in ["exact_duplicate_rows", "duplicate_texts_exact", "duplicate_texts_normalized", "conflicting_normalized_text_groups"]}

## Visualize distributions

The generated HTML contains category, intent and word-length distribution bars. Counts exclude rows missing required input or target values.

In [ ]:
write_report(summary, ROOT / "reports")
from IPython.display import HTML, display
display(HTML((ROOT / "reports/eda_report.html").read_text(encoding="utf-8")))

## Findings and next decisions

The initial audit found 26,872 rows, 11 categories and 27 intents. There are no missing fields. There are 2,598 repeated messages beyond the first occurrence after case/whitespace normalization, without conflicting labels.

Before training: deduplicate, create reproducible splits stratified by intent, and assert no normalized message overlap. Inspect template similarity separately; exact deduplication does not eliminate paraphrase leakage. Use only `instruction` as input. Each intent maps to one category in this revision, allowing category derivation from intent.

Questions: Why do full-row duplicate counts differ from message duplicate counts? Why is class balance different for intents and categories? Why should we fit TF-IDF only on the training split?